# Visualize Task04 Results
Load `data/experiment/cf_exp_recon.npz` and visualize vn / vs ground truth and reconstruction.

In [4]:
import numpy as np
from sf_recon.utils.io import load_npz
print('Ready to load Task04 results')

Ready to load Task04 results


In [5]:
import numpy as np

result_path = '../data/experiment/cf_exp_recon.npz'
with np.load(result_path, allow_pickle=True) as data:
    print('=== NPZ keys ===')
    print(sorted(data.files))

    for key in ['gt', 'recon', 'loss_mask', 'reset_mask', 'v_gt_speed', 'v_recon_speed']:
        value = data.get(key)
        if value is None:
            print(f'{key}: None')
        else:
            arr = np.asarray(value)
            print(f'{key}: shape={arr.shape}, dtype={arr.dtype}')

    gt = data.get('gt')
    recon = data.get('recon')
    loss_mask = data.get('loss_mask')

def _safe_array(x):
    if x is None:
        return None
    try:
        return np.asarray(x)
    except Exception:
        return None

gt_arr = _safe_array(gt)
recon_arr = _safe_array(recon)
mask_arr = _safe_array(loss_mask)

print('\n=== Marker preview ===')
for name, arr in [('gt', gt_arr), ('recon', recon_arr), ('loss_mask', mask_arr)]:
    if arr is None:
        print(f'{name}: unavailable')
    else:
        print(f'{name}: ndim={arr.ndim}, shape={arr.shape}')

def _visible_count(points, mask, frame_idx):
    if points is None or points.ndim != 3 or frame_idx >= points.shape[0]:
        return None
    p = np.asarray(points[frame_idx])
    if p.ndim != 2 or p.shape[-1] != 2:
        return None
    if mask is None or mask.ndim < 2 or frame_idx >= mask.shape[0]:
        return len(p)
    m = np.asarray(mask[frame_idx]).reshape(-1) > 0.5
    if len(m) != len(p):
        return f'mask-mismatch points={len(p)} mask={len(m)}'
    return int(np.sum(m))

for i in range(3):
    gt_count = _visible_count(gt_arr, mask_arr, i)
    recon_count = _visible_count(recon_arr, mask_arr, i)
    print(f'frame {i}: gt_visible={gt_count}, recon_visible={recon_count}')

=== NPZ keys ===
['Lx', 'Ly', 'Wx', 'Wy', 'gt', 'gt_vel', 'loss_mask', 'marker_x_max', 'marker_x_min', 'marker_y_max', 'marker_y_min', 'pre_steps', 'recon', 'recon_vel', 'reset_mask', 'success', 't_gt', 't_recon', 'v_gt_speed', 'v_gt_u', 'v_gt_v', 'v_gt_vec', 'v_recon_speed', 'v_recon_u', 'v_recon_v', 'v_recon_vec', 'vel_mask', 'vs_gt_speed', 'vs_gt_u', 'vs_gt_v', 'vs_gt_vec', 'vs_recon_speed', 'vs_recon_u', 'vs_recon_v', 'vs_recon_vec', 'window_x_max', 'window_x_min', 'window_y_max', 'window_y_min']
gt: shape=(7, 8, 2), dtype=float64
recon: shape=(7, 8, 2), dtype=float64
loss_mask: shape=(7, 8), dtype=float64
reset_mask: shape=(7, 8), dtype=float64
v_gt_speed: shape=(7, 320, 16), dtype=float64
v_recon_speed: shape=(7, 320, 16), dtype=float64

=== Marker preview ===
gt: ndim=3, shape=(7, 8, 2)
recon: ndim=3, shape=(7, 8, 2)
loss_mask: ndim=2, shape=(7, 8)
frame 0: gt_visible=8, recon_visible=8
frame 1: gt_visible=5, recon_visible=5
frame 2: gt_visible=3, recon_visible=3


In [6]:
# ================================================================================
# Velocity fields visualization (Task04 Experimental Counterflow)
# Line 1: vn-GT、vs-GT
# Line 2: vn-Recon、vs-Recon
# ================================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import matplotlib as mpl
from matplotlib.colors import ListedColormap

bmap = mpl.cm.twilight_shifted
rmap = mpl.cm.twilight
bluecolors = bmap(np.linspace(0, 1, 256))
redcolors = rmap(np.linspace(0, 1, 256))
cblue = ListedColormap(bluecolors[:100])
cred = ListedColormap(redcolors[128:228])

result_path = '../data/experiment/cf_exp_recon.npz'

with np.load(result_path, allow_pickle=True) as data:
    Lx = float(data['Lx']) if 'Lx' in data else 0.016
    Ly = float(data['Ly']) if 'Ly' in data else 0.320
    Wx = float(data['Wx']) if 'Wx' in data else 0.016
    Wy = float(data['Wy']) if 'Wy' in data else 0.008

    saved_window_x_min = float(data['window_x_min']) if 'window_x_min' in data else 0.0
    saved_window_x_max = float(data['window_x_max']) if 'window_x_max' in data else Wx
    saved_window_y_min = float(data['window_y_min']) if 'window_y_min' in data else (Ly / 2 - Wy / 2)
    saved_window_y_max = float(data['window_y_max']) if 'window_y_max' in data else (Ly / 2 + Wy / 2)
    saved_marker_x_min = float(data['marker_x_min']) if 'marker_x_min' in data else None
    saved_marker_x_max = float(data['marker_x_max']) if 'marker_x_max' in data else None
    saved_marker_y_min = float(data['marker_y_min']) if 'marker_y_min' in data else None
    saved_marker_y_max = float(data['marker_y_max']) if 'marker_y_max' in data else None

    v_gt_speed = data.get('v_gt_speed')
    v_recon_speed = data.get('v_recon_speed')
    v_gt_u = data.get('v_gt_u')
    v_gt_v = data.get('v_gt_v')
    v_recon_u = data.get('v_recon_u')
    v_recon_v = data.get('v_recon_v')
    vs_gt_speed = data.get('vs_gt_speed')
    vs_recon_speed = data.get('vs_recon_speed')
    vs_gt_u = data.get('vs_gt_u')
    vs_gt_v = data.get('vs_gt_v')
    vs_recon_u = data.get('vs_recon_u')
    vs_recon_v = data.get('vs_recon_v')
    gt = data.get('gt')
    recon = data.get('recon')
    loss_mask = data.get('loss_mask')

def as_time_array(x):
    if x is None:
        return None
    try:
        a = np.array(x)
    except Exception:
        try:
            a = np.array(list(x))
        except Exception:
            return None
    if a.dtype == object:
        try:
            a = np.stack([np.asarray(e, dtype=float) for e in a])
        except Exception:
            return None
    try:
        return a.astype(float)
    except Exception:
        return None

def normalize_marker_series(x):
    a = as_time_array(x)
    if a is None:
        return None
    if a.ndim == 3 and a.shape[-1] == 2:
        return a
    if a.ndim == 3 and a.shape[1] == 2:
        return np.transpose(a, (0, 2, 1))
    if a.ndim == 2 and a.shape[-1] == 2:
        return a[None, ...]
    if a.ndim == 1 and a.size % 2 == 0:
        try:
            return a.reshape(1, -1, 2)
        except Exception:
            return None
    return None

def normalize_mask_series(x, target_time=None, target_markers=None):
    a = as_time_array(x)
    if a is None:
        return None
    if a.ndim == 2:
        pass
    elif a.ndim == 1:
        a = a[None, ...]
    else:
        return None
    if target_time is not None and a.shape[0] != target_time:
        if a.shape[1] == target_time and (target_markers is None or a.shape[0] == target_markers):
            a = a.T
    return a.astype(float)

v_gt_speed = as_time_array(v_gt_speed)
v_recon_speed = as_time_array(v_recon_speed)
v_gt_u = as_time_array(v_gt_u)
v_gt_v = as_time_array(v_gt_v)
v_recon_u = as_time_array(v_recon_u)
v_recon_v = as_time_array(v_recon_v)
vs_gt_speed = as_time_array(vs_gt_speed)
vs_recon_speed = as_time_array(vs_recon_speed)
vs_gt_u = as_time_array(vs_gt_u)
vs_gt_v = as_time_array(vs_gt_v)
vs_recon_u = as_time_array(vs_recon_u)
vs_recon_v = as_time_array(vs_recon_v)
gt = normalize_marker_series(gt)
recon = normalize_marker_series(recon)

T_candidates = []
for arr in (v_gt_speed, v_recon_speed, v_gt_u, v_recon_u, vs_gt_speed, vs_recon_speed, gt, recon):
    if arr is not None:
        try:
            T_candidates.append(int(np.asarray(arr).shape[0]))
        except Exception:
            pass
T = max(T_candidates) if T_candidates else 1

marker_count = None
for arr in (gt, recon):
    if arr is not None and arr.ndim == 3:
        marker_count = arr.shape[1]
        break
loss_mask = normalize_mask_series(loss_mask, target_time=T, target_markers=marker_count)

def select_points(pts, mask, i):
    if pts is None or pts.ndim != 3:
        return None
    if i >= pts.shape[0]:
        return None
    p = np.asarray(pts[i], dtype=float)
    if p.ndim != 2 or p.shape[1] != 2:
        return None
    if mask is not None and i < mask.shape[0]:
        try:
            visible = np.asarray(mask[i]).reshape(-1) > 0.5
            if visible.shape[0] == p.shape[0]:
                p = p[visible]
        except Exception:
            pass
    if p.size == 0:
        return None
    return p

def compute_marker_bounds(*marker_arrays):
    xs = []
    ys = []
    for arr in marker_arrays:
        if arr is None:
            continue
        pts = np.asarray(arr, dtype=float)
        if pts.ndim != 3 or pts.shape[-1] != 2:
            continue
        valid = np.isfinite(pts[..., 0]) & np.isfinite(pts[..., 1])
        if not np.any(valid):
            continue
        xs.append(pts[..., 0][valid])
        ys.append(pts[..., 1][valid])
    if not xs or not ys:
        return None
    x_all = np.concatenate(xs)
    y_all = np.concatenate(ys)
    return float(np.min(x_all)), float(np.max(x_all)), float(np.min(y_all)), float(np.max(y_all))

auto_bounds = compute_marker_bounds(gt, recon)
use_saved_marker_bounds = None not in (saved_marker_x_min, saved_marker_x_max, saved_marker_y_min, saved_marker_y_max)
if use_saved_marker_bounds:
    marker_x_min, marker_x_max = saved_marker_x_min, saved_marker_x_max
    marker_y_min, marker_y_max = saved_marker_y_min, saved_marker_y_max
elif auto_bounds is not None:
    marker_x_min, marker_x_max, marker_y_min, marker_y_max = auto_bounds
else:
    marker_x_min, marker_x_max = saved_window_x_min, saved_window_x_max
    marker_y_min, marker_y_max = saved_window_y_min, saved_window_y_max

marker_pad_y = max(0.0005, 0.10 * max(marker_y_max - marker_y_min, 1e-6))
effective_y_min = max(0.0, marker_y_min - marker_pad_y)
effective_y_max = min(Ly, marker_y_max + marker_pad_y)
if effective_y_max <= effective_y_min:
    effective_y_min, effective_y_max = saved_window_y_min, saved_window_y_max

effective_x_min = saved_window_x_min
effective_x_max = saved_window_x_max

X = Y = None
u_shape = None
for arr in (v_gt_u, v_recon_u, vs_gt_u, vs_recon_u):
    if arr is not None:
        u_shape = arr.shape
        break
if u_shape is not None and len(u_shape) >= 3:
    H, W = u_shape[1], u_shape[2]
    x = np.linspace(0, Lx, W)
    y = np.linspace(0, Ly, H)
    X, Y = np.meshgrid(x, y)
    y_idx = np.where((y >= effective_y_min) & (y <= effective_y_max))[0]
else:
    y = None
    y_idx = None

def get_local_min_max(speed_field, y_indices):
    if speed_field is None:
        return 0.0, 1.0
    if y_indices is None or len(y_indices) == 0:
        local_data = speed_field
    else:
        local_data = speed_field[:, y_indices, :]
    _min, _max = np.nanmin(local_data), np.nanmax(local_data)
    return _min, _max

vmin_gt, vmax_gt = get_local_min_max(v_gt_speed, y_idx)
vmin_re, vmax_re = get_local_min_max(v_recon_speed, y_idx)
vs_vmin_gt, vs_vmax_gt = get_local_min_max(vs_gt_speed, y_idx)
vs_vmin_re, vs_vmax_re = get_local_min_max(vs_recon_speed, y_idx)

fig, axs = plt.subplots(2, 2, figsize=(20, 8), constrained_layout=True)
fig.subplots_adjust(wspace=0.18, hspace=0.18)
titles = ['vn-GT', 'vs-GT', 'vn-Recon', 'vs-Recon']

def clip_points_to_axes(points):
    if points is None or points.size == 0:
        return None, 0
    visible = (
        (points[:, 0] >= effective_x_min) & (points[:, 0] <= effective_x_max) &
        (points[:, 1] >= effective_y_min) & (points[:, 1] <= effective_y_max)
    )
    clipped = points[visible]
    return (clipped if clipped.size else None), int(np.sum(visible))



def draw_frame(ax, i, speed_arr, u_arr, v_arr, pts, title, cmap, mask=None, vmin=None, vmax=None):
    ax.clear()
    ax.set_xlim(effective_x_min, effective_x_max)
    ax.set_ylim(effective_y_min, effective_y_max)
    ax.set_aspect(6/19)
    ax.set_facecolor('black')
    if speed_arr is not None and i < speed_arr.shape[0]:
        try:
            data = np.asarray(speed_arr[i], dtype=float)
            im = ax.imshow(
                data,
                origin='lower',
                extent=[0, Lx, 0, Ly],
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                aspect='auto',
                interpolation='bilinear',
                zorder=1,
            )
            if not hasattr(ax, '_has_cbar'):
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label('speed')
                ax._has_cbar = True
        except Exception:
            pass
    if u_arr is not None and v_arr is not None and X is not None and i < u_arr.shape[0] and i < v_arr.shape[0]:
        try:
            u = np.asarray(u_arr[i], dtype=float)
            v = np.asarray(v_arr[i], dtype=float)
            ax.streamplot(X, Y, u, v, color='whitesmoke', linewidth=1.0, arrowsize=1.5, density=0.8)
        except Exception:
            pass
    raw_points = select_points(pts, mask, i)
    points, inside_count = clip_points_to_axes(raw_points)
    total_count = 0 if raw_points is None else len(raw_points)
    if points is not None:
        try:
            ax.scatter(points[:, 0], points[:, 1], s=180, marker='x', c='#FFFF00', linewidths=2.0, alpha=1.0, zorder=10)
            ax.scatter(points[:, 0], points[:, 1], s=70, marker='o', c='lightblue', edgecolors='white', linewidths=1.0, alpha=1.0, zorder=11)
        except Exception:
            pass
    ax.axhline(saved_window_y_min, color='cyan', linestyle='--', linewidth=0.8, alpha=0.8, zorder=10)
    ax.axhline(saved_window_y_max, color='cyan', linestyle='--', linewidth=0.8, alpha=0.8, zorder=10)
    ax.set_title(f'{title} t={i}')
    ax.set_xlabel('x [m]')
    ax.set_ylabel('y [m]')

def update(i):
    draw_frame(axs[0, 0], i, v_gt_speed, v_gt_u, v_gt_v, gt, titles[0], cred, loss_mask, vmin_gt, vmax_gt)
    draw_frame(axs[0, 1], i, vs_gt_speed, vs_gt_u, vs_gt_v, gt, titles[1], cblue, loss_mask, vs_vmin_gt, vs_vmax_gt)
    draw_frame(axs[1, 0], i, v_recon_speed, v_recon_u, v_recon_v, recon, titles[2], cred, loss_mask, vmin_re, vmax_re)
    draw_frame(axs[1, 1], i, vs_recon_speed, vs_recon_u, vs_recon_v, recon, titles[3], cblue, loss_mask, vs_vmin_re, vs_vmax_re)
    return []

num_frames = min(20, T)
anim = animation.FuncAnimation(fig, update, frames=num_frames, interval=300, blit=False)
plt.close(fig)
anim.save('../data/experiment/cf_exp_recon.gif')
HTML(anim.to_jshtml())

C:\Users\XU_li\AppData\Local\Temp\ipykernel_30792\1107175464.py:227: UserWarning: This figure was using a layout engine that is incompatible with subplots_adjust and/or tight_layout; not calling subplots_adjust.
  fig.subplots_adjust(wspace=0.18, hspace=0.18)
MovieWriter ffmpeg unavailable; using Pillow instead.
